<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module12/Lab2.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 2 — From Quantum Gates to QAOA Parameters (Instructor)
**Quantum Optimization and Simulation — QAOA Laboratory Series**

Instructor version with completed exercises and answer key.

**Format:** 10–15 minute instructor walkthrough + about 45–60 minutes of independent work.

**Notebook style:** Most code is supplied. Cells marked **YOUR TURN** contain a small value, line, or function for you to complete.

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- See that \(R_Z(\gamma)\) changes relative phase without directly changing Z-basis probabilities.
- See that \(R_X(\beta)\) can turn phase information into different measurement probabilities.
- Connect the parameters \(\gamma\) and \(eta\) to the QAOA ideas of **cost** and **mixer**.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

## Part A — Start with everyone 'on stage': \(H|0\rangle=|+\rangle\)

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.measure_all()
counts = run_counts(qc)
print(counts)
plot_histogram(counts)

**Expected:** approximately 50% `0` and 50% `1`.

## Part B — Add \(R_Z(\gamma)\)

In [ ]:
gamma = np.pi / 3

qc = QuantumCircuit(1)
qc.h(0)
qc.rz(gamma, 0)
qc.measure_all()

counts = run_counts(qc)
print(counts)
plot_histogram(counts)

**Observation:** The Z-basis histogram should still be close to 50/50. \(R_Z\) records information in **phase**, which is not directly visible in a Z-basis measurement.

## Part C — Add the mixer \(R_X\)

In [ ]:
gamma = np.pi / 3
beta = np.pi / 8

qc = QuantumCircuit(1)
qc.h(0)
qc.rz(gamma, 0)

# Qiskit defines RX(theta)=exp(-i theta X / 2).
# QAOA UB=exp(-i beta X), therefore the Qiskit angle is 2*beta.
qc.rx(2 * beta, 0)

qc.measure_all()
counts = run_counts(qc)
print(counts)
plot_histogram(counts)

### YOUR TURN
Keep `gamma = pi/3`. Try:
- `beta = 0`
- `beta = pi/16`
- `beta = pi/8`
- `beta = pi/4`

Record \(P(0)\) and \(P(1)\).

**Question:** Why can \(R_X\) change the measured probabilities even though \(R_Z\) alone did not?

In [ ]:
beta_values = [0, np.pi/16, np.pi/8, np.pi/4]

results = []
for beta in beta_values:
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.rz(np.pi/3, 0)

    # TODO: add the QAOA-style mixer rotation here.
    # qc.rx( ______ , 0)

    qc.measure_all()
    counts = run_counts(qc, shots=4096)
    total = sum(counts.values())
    p0 = counts.get("0", 0) / total
    p1 = counts.get("1", 0) / total
    results.append((beta, p0, p1))

results

**Expected qualitative output:** `beta=0` remains close to 50/50. Other beta values generally move probability between `0` and `1`.

## Take-home question
Complete the sentence:

> In QAOA, the cost operation primarily puts information into ________, and the mixer allows that information to affect ________.

## Instructor solutions

In [ ]:
beta_values = [0, np.pi/16, np.pi/8, np.pi/4]

results = []
for beta in beta_values:
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.rz(np.pi/3, 0)
    qc.rx(2 * beta, 0)
    qc.measure_all()

    counts = run_counts(qc, shots=4096)
    total = sum(counts.values())
    p0 = counts.get("0", 0) / total
    p1 = counts.get("1", 0) / total
    results.append((beta, p0, p1))

results

**Conceptual answer:** \(R_Z\) creates a relative phase. \(R_X\) rotates in a different basis, causing amplitudes to interfere, so the phase difference can become a population/probability difference.

**Fill-in:** phase; measurement probabilities (amplitudes/populations).